In [ ]:
import numpy as np
import pandas as pd

# 准备数据
- 每个大洲的原始SWOT数据，需要进行一些data filter
- 如果每次计算的时候都执行filter，会花费很多时间，所以将filter后的文件存起来。微调算法的时候可以直接读
- 这一部分可以进行很多测试，保存执行了不同filter的数据文件，以硬盘空间换计算时间

In [ ]:
# 对每个reach进行筛选，仅保留参与 node 数大于 0.9N 的日期
# 原因：某些node数据被filter之后，在reach内保留node数量太少；
# 并不足以形成足够稳定的profile，对median profile的负面影响太大
def filter_valid_dates(group):
    # 统计 unique node_id 的数量 N
    unique_nodes = group['node_id'].nunique()
    threshold = 0.9 * unique_nodes
    
    # 计算每个日期的 unique node_id 数量
    date_counts = group.groupby('date')['node_id'].nunique()
    
    # 找出参与 node 数大于 0.9N 的日期
    valid_dates = date_counts[date_counts > threshold].index
    
    # 如果 valid_dates 的数量小于 5，则返回空 DataFrame
    if len(valid_dates) < 5:
        return pd.DataFrame()
    
    # 筛选出这些日期的数据
    valid_data = group[group['date'].isin(valid_dates)]
    
    return valid_data

In [ ]:
name_list=['af','eu','si','as','au','sa','na','ar','gr']

# 逐大洲处理
for name in name_list:
    # 使用合适的数据类型加载数据
    df = pd.read_csv(
        f'D:/HC_P_data/RiverSP_s/cycle1/node_{name}.csv',)

    # 根据node_id计算 reach_id
    df['reach_id'] = df['node_id'] // 10000 * 10 + df['node_id'] % 10

    # 筛选条件
    df = df[(df['wse_u'] < 0.5) & (df['ice_clim_f'] == 0)]

    # 河流长度 >= 10
    valid_reach_ids = df.groupby('reach_id').filter(lambda x: len(x) >= 10)['reach_id'].unique()
    df = df[df['reach_id'].isin(valid_reach_ids)]

    # 河流类型 = 1（river类型）
    df['reach_type'] = df['reach_id'] % 10
    df = df[df['reach_type'] == 1]

    # 转换时间格式
    df['time_str'] = pd.to_datetime(df['time'], unit='s', origin='2000-01-01')
    df['date'] = df['time_str'].dt.date

    # 根据上面filter_valid_dates函数的要求，筛选每个 reach_id 内的数据
    df = df.groupby('reach_id').apply(filter_valid_dates).reset_index(drop=True)

    # 选择需要的列并写入 CSV
    df[['node_id', 'time', 'wse', 'width', 'ice_clim_f', 'dark_frac', 'rdr_sig0', 'node_q_b', 'xovr_cal_q', 'wse_u']].to_csv(
        f'D:/HC_P_data/RiverSP_s/cycle1/node_{name}_noice_u.csv',# 命名代表：没有河冰，还对wse_u进行了筛选
        index=False
    )

    print(name)

In [ ]:
# 计算每个reach，有node参与的月份数
# 原因：某些node数据被filter之后，只在一年中有限几个月有数据
# 这种情况生成的profile也不太靠谱
def calc_poor_coverage_ratio(group):
    total_nodes = group.shape[0] # group 已经为每个 node 一行
    # 定义阈值：出现月份数少于4个月的node就是poor_count
    threshold = 4
    poor_count = (group['month_count'] <= threshold).sum()
    return poor_count / total_nodes# 输出：poor_count的node占该reach的总node数

In [ ]:
%%time
name_list=['af','eu','si','as','au','sa','na','ar','gr']
for name in name_list:
    df=pd.read_csv('D:/HC_P_data/RiverSP_s/cycle1/node_'+name+'_noice_u.csv')
    df['time_str'] = pd.to_datetime(df['time'], unit='s', origin='2000-01-01')
    df['date']=df['time_str'].dt.date
    df['date'] = pd.to_datetime(df['date'])
    df['reach_id']=df['node_id']//10000*10+df['node_id']%10
    df['month'] = df['date'].dt.month

    node_month_counts = df.groupby(['reach_id', 'node_id'])['month'].nunique().reset_index(name='month_count')
    
    reach_ratio = node_month_counts.groupby('reach_id').apply(calc_poor_coverage_ratio).reset_index(name='poor_coverage_ratio')
    # 如果某reach，poor_count的node占到了总node数的80%以上，则认为该reach数据覆盖率太差，去掉
    keep_reaches = reach_ratio.loc[reach_ratio['poor_coverage_ratio'] < 0.8, 'reach_id']
    filtered_df = df[df['reach_id'].isin(keep_reaches)]
    filtered_df.to_csv('D:/HC_P_data/RiverSP_s/cycle1/node_'+name+'_noice_u_month4.csv')# month4代表上面calc_poor_coverage_ratio函数的阈值是4
    print(name)

# SWOOP算法

In [ ]:
def detect_step_change(denoised_signal, sig_signal, occu_signal, std_signal, 
                       stability_scale=0.2, change_ratio_thres=0.1, sigma_thres=10, 
                       occu_thres=30,time_thres=1):
    # 计算WSE的差分
    differences = np.abs(np.diff(denoised_signal))
    step_points = np.where(differences > 0.5)[0] + 1# 大于0.5m的高差才被考虑
    window_size1=10
    window_size2=10
    
    if len(step_points) > 0:
        step_index = step_points[np.argmax(differences[step_points - 1])]# 选取最大的差分点作为潜在的step
        # 保证step_index前后有足够分析的窗口（5-10个）
        if step_index < window_size1 or step_index >= len(denoised_signal) - window_size2:
            window_size1=min(window_size1,step_index)
            window_size2=min(window_size2, len(denoised_signal) - step_index)
        if window_size1<=5 or window_size2<=5:
            return None, False

        # 提取前后窗口的信号段
        before_segment = denoised_signal[step_index - window_size1:step_index]
        after_segment = denoised_signal[step_index+1:step_index + window_size2]
        
        # change_size1:相邻节点的局部差
        change_size1 = differences[step_index - 1]
        
        # 对前后段进行线性拟合，计算残差
        before_trend_coeff = np.polyfit(range(window_size1), before_segment, 1)
        after_trend_coeff = np.polyfit(range(window_size2-1), after_segment, 1)
        before_residuals = before_segment - np.polyval(before_trend_coeff, range(window_size1))
        after_residuals = after_segment - np.polyval(after_trend_coeff, range(window_size2-1))
        # 计算残差的标准差
        before_residual_std = np.std(before_residuals)
        after_residual_std = np.std(after_residuals)

        # change_size2:前后段线性拟合后的局部差
        change_size2 = abs(np.polyval(before_trend_coeff, range(window_size1+1))[-1]-np.polyval(after_trend_coeff, range(window_size2-1))[0])
        change_size=min(change_size1,change_size2)# 取较小的那个
        
        # 判断突变前后是否稳定
        stability_threshold = stability_scale * change_size

        # 突变高差与整体reach的高差的比例（还是保留了这个，不然西伯利亚有很多False Positive）
        range_=np.max(denoised_signal)-np.min(denoised_signal)
        if change_size/range_<change_ratio_thres:
            return None, False

        # 前后窗口内的平均后向散射系数
        sig=np.mean(list(sig_signal[step_index-5:step_index])+list(sig_signal[step_index+1:step_index+5]))
        if sig<sigma_thres:
            return None, False

        # 前后窗口内的平均water occurrence
        occu=np.nanmean(list(occu_signal[step_index-5:step_index])+list(occu_signal[step_index+1:step_index+5]))
        if occu<occu_thres:
            return None, False

        # SWOT观测数据的temporal std
        time_std=np.mean(list(std_signal[step_index-5:step_index])+list(std_signal[step_index+1:step_index+5]))/(range_)
        if time_std>time_thres:
            return None, False

        # 前面都是filter条件
        # 这里是稳定性检查
        if before_residual_std < stability_threshold and after_residual_std < stability_threshold:
            return [step_index,change_size1,change_size2, range_,sig,occu,time_std,before_residual_std, after_residual_std], True

    return None, False

In [ ]:
# 这里是保存了每个node的后向散射系数的年平均
df_sig=pd.read_csv('D:/HC_P_data/RiverSP_s/cycle1/node_sig_mean.csv')
# 这个是每个node的water occurrence文件
df_occu=pd.read_csv('D:/HC_P_data/RiverSP_s/node_occu_mean.csv')
df_occu.loc[df_occu['occurrence'] == 255, 'occurrence'] = 0

In [ ]:
# 测试的是没有进行poor_coverage_ratio筛选的例子
name_list=['af','eu','si','as','au','sa','na','ar','gr']
for name in name_list:
    df=pd.read_csv('D:/HC_P_data/RiverSP_s/cycle1/node_'+name+'_noice_u.csv')
    df['reach_id']=df['node_id']//10000*10+df['node_id']%10
    
    # 计算每个 node_id 的 median
    average_df=df.groupby(['node_id'])['wse'].median().reset_index(name='wse_median')
    # 计算每个 node_id 的std
    std_df = df.groupby(df['node_id'])['wse'].std().reset_index(name='wse_std')
    average_df=average_df.merge(std_df,on='node_id')
    
    # sigma
    average_df=average_df.merge(df_sig,on='node_id')
    # occurrence
    average_df=average_df.merge(df_occu,on='node_id')
    
    # 按 node_id 排序
    average_df = average_df.sort_values(by='node_id')
    average_df['reach_id']=average_df['node_id']//10000*10+average_df['node_id']%10
    
    results = []
    for reach_id, group in average_df.groupby('reach_id'):
        signal = group['wse_median'].values
        infos, has_step = detect_step_change(signal,group['rdr_sig0'].values,group['occurrence'].values,group['wse_std'].values,
                                        stability_scale=0.2, change_ratio_thres=0.1, 
                                        sigma_thres=20, occu_thres=30,time_thres=1)
        if has_step:
            results.append({'reach_id': reach_id,
                            'node_id':group['node_id'].iloc[infos[0]],
                            'step_index':infos[0],
                            'change_size1':infos[1],
                            'change_size2':infos[2],
                            'change_ratio':infos[2]/infos[3],
                            'sigma':infos[4],
                            'occu':infos[5],
                            'time_std':infos[6],
                            'before_std':infos[7],
                            'after_std':infos[8]
                            })
    step_df = pd.DataFrame(results)
    step_df.to_csv('D:/HC_P_data/csv_result/'+name+'-valid-1.csv')
    print(name)

In [ ]:
# 全球的合并到一起（文件不大）
name_list=['af','eu','si','as','au','sa','na','ar','gr']
dfs=[]
for name in name_list:
    df=pd.read_csv('D:/HC_P_data/csv_result/'+name+'-valid-1.csv')
    dfs.append(df)
df=pd.concat(dfs)
df=df.sort_values(by='node_id')# 按照node排序之后更规整
df.to_csv('D:/HC_P_data/csv_result/obstruction_v1-check.csv',index=False)

In [ ]:
# 测试的是进行过poor_coverage_ratio筛选的例子
name_list=['af','eu','si','as','au','sa','na','ar','gr']
for name in name_list:
    df=pd.read_csv('D:/HC_P_data/RiverSP_s/cycle1/node_'+name+'_noice_u_month4.csv')# 换一个输入源的事
    df['reach_id']=df['node_id']//10000*10+df['node_id']%10
    
    # 计算每个 node_id 的 median
    average_df=df.groupby(['node_id'])['wse'].median().reset_index(name='wse_median')
    # 计算每个 node_id 的std
    std_df = df.groupby(df['node_id'])['wse'].std().reset_index(name='wse_std')
    average_df=average_df.merge(std_df,on='node_id')
    
    # sigma
    average_df=average_df.merge(df_sig,on='node_id')
    # occurrence
    average_df=average_df.merge(df_occu,on='node_id')
    
    # 按 node_id 排序
    average_df = average_df.sort_values(by='node_id')
    average_df['reach_id']=average_df['node_id']//10000*10+average_df['node_id']%10
    
    results = []
    for reach_id, group in average_df.groupby('reach_id'):
        signal = group['wse_median'].values
        infos, has_step = detect_step_change(signal,group['rdr_sig0'].values,group['occurrence'].values,group['wse_std'].values,
                                        stability_scale=0.2, change_ratio_thres=0.1, 
                                        sigma_thres=20, occu_thres=30,time_thres=1)
        if has_step:
            results.append({'reach_id': reach_id,
                            'node_id':group['node_id'].iloc[infos[0]],
                            'step_index':infos[0],
                            'change_size1':infos[1],
                            'change_size2':infos[2],
                            'change_ratio':infos[2]/infos[3],
                            'sigma':infos[4],
                            'occu':infos[5],
                            'time_std':infos[6],
                            'before_std':infos[7],
                            'after_std':infos[8]
                            })
    step_df = pd.DataFrame(results)
    step_df.to_csv('D:/HC_P_data/csv_result/'+name+'-valid-1.2.csv')
    print(name)

In [ ]:
# 全球的合并到一起
name_list=['af','eu','si','as','au','sa','na','ar','gr']
dfs=[]
for name in name_list:
    df=pd.read_csv('D:/HC_P_data/csv_result/'+name+'-valid-1.2.csv')
    dfs.append(df)
df=pd.concat(dfs)
df=df.sort_values(by='node_id')# 按照node排序之后更规整
df.to_csv('D:/HC_P_data/csv_result/obstruction_v1.2-check.csv',index=False)